In [1]:
 
import pandas as pd
import sqlite3

print("Libraries loaded ✅")

Libraries loaded ✅


In [2]:
df = pd.read_excel("../data/Bank_Personal_Loan_Modelling.xlsx")

print("Shape:", df.shape)
df.head()

Shape: (5000, 14)


,ID,Age,Experience,Income,ZIP Code,Family,CCAvg,Education,Mortgage,Personal Loan,Securities Account,CD Account,Online,CreditCard
0,1,25,1,49,91107,4,1960-01-01 00:00:00,1,0,0,1,0,0,0
1,2,45,19,34,90089,3,1950-01-01 00:00:00,1,0,0,1,0,0,0
2,3,39,15,11,94720,1,2000-01-01 00:00:00,1,0,0,0,0,0,0
3,4,35,9,100,94112,1,1970-02-01 00:00:00,2,0,0,0,0,0,0
4,5,35,8,45,91330,4,2000-01-01 00:00:00,2,0,0,0,0,0,1


In [3]:
print("Data Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

Data Types:
ID                     int64
Age                    int64
Experience             int64
Income                 int64
ZIP Code               int64
Family                 int64
CCAvg                 object
Education              int64
Mortgage               int64
Personal Loan          int64
Securities Account     int64
CD Account             int64
Online                 int64
CreditCard             int64
dtype: object

Missing Values:
ID                    0
Age                   0
Experience            0
Income                0
ZIP Code              0
Family                0
CCAvg                 0
Education             0
Mortgage              0
Personal Loan         0
Securities Account    0
CD Account            0
Online                0
CreditCard            0
dtype: int64


In [4]:
df.drop(columns=["ID", "ZIP Code"], inplace=True)

print("Remaining columns:", df.columns.tolist())

Remaining columns: ['Age', 'Experience', 'Income', 'Family', 'CCAvg', 'Education', 'Mortgage', 'Personal Loan', 'Securities Account', 'CD Account', 'Online', 'CreditCard']


In [5]:
def fix_ccavg(val):
    try:
        if isinstance(val, pd.Timestamp):
            # date like 1970-02-01 → month=2, day=1 → 2.1
            return round(val.month + val.day / 100, 2)
        if isinstance(val, str) and "/" in val:
            # string like "0/40" → take first number → 0
            return float(val.split("/")[0])
        return float(val)
    except:
        return 0.0

df["CCAvg"] = df["CCAvg"].apply(fix_ccavg)

print("CCAvg fixed ✅")
print("Sample values:", df["CCAvg"].head(5).tolist())
print("Data type now:", df["CCAvg"].dtype)

CCAvg fixed ✅
Sample values: [0.0, 0.0, 0.0, 0.0, 0.0]
Data type now: float64


In [6]:
negative_count = (df["Experience"] < 0).sum()
print(f"Negative experience rows found: {negative_count}")

df["Experience"] = df["Experience"].clip(lower=0)

print("Fixed ✅ — all values now >= 0")

Negative experience rows found: 52
Fixed ✅ — all values now >= 0


In [7]:
print("Clean Data Shape:", df.shape)
print("\nMissing Values:", df.isnull().sum().sum())
print("\nTarget column (Personal Loan):")
print(df["Personal Loan"].value_counts())
print("\nSample:")
df.head()

Clean Data Shape: (5000, 12)

Missing Values: 0

Target column (Personal Loan):
Personal Loan
0    4520
1     480
Name: count, dtype: int64

Sample:


,Age,Experience,Income,Family,CCAvg,Education,Mortgage,Personal Loan,Securities Account,CD Account,Online,CreditCard
0,25,1,49,4,0.0,1,0,0,1,0,0,0
1,45,19,34,3,0.0,1,0,0,1,0,0,0
2,39,15,11,1,0.0,1,0,0,0,0,0,0
3,35,9,100,1,0.0,2,0,0,0,0,0,0
4,35,8,45,4,0.0,2,0,0,0,0,0,1


In [8]:
conn = sqlite3.connect("../data/bank_loan.db")
df.to_sql("clean_data", conn, if_exists="replace", index=False)
conn.close()

print("✅ Clean data saved to: data/bank_loan.db")
print(f"   Table: clean_data | Rows: {len(df)} | Columns: {len(df.columns)}")

✅ Clean data saved to: data/bank_loan.db
   Table: clean_data | Rows: 5000 | Columns: 12


- Loaded 5000 rows from Excel
- Fixed broken `CCAvg` column
- Dropped `ID` and `ZIP Code`
- Fixed negative `Experience` values
- Saved clean data to SQLite database